In [1]:
import torch
import torch.nn.functional as F
import pandas as pd

In [2]:
df = pd.read_csv("dataset.csv")
X_fen = df["FEN"]
Y = df["EVAL"]

In [3]:
def fen_to_pos(fen):
    pos = [0] * (12 * 64 + 1 + 4 + 1)
    fen_parts = fen.split(' ')
    
    row = 7
    col = 0
    piece_to_int = {'K':0, 'Q':1, 'R':2, 'B':3, 'N':4, 'P':5, 
                    'k':6, 'q':7, 'r':8, 'b':9, 'n':10, 'p':11}
    
    for c in fen_parts[0]:
        if c == '/':
            row -= 1
            col = 0
        elif c.isdigit():
            col += int(c)
        else:
            piece_idx = piece_to_int[c]
            pos[piece_idx * 64 + row * 8 + col] = 1
            col += 1

    if fen_parts[1] == 'b':
        pos[768] = 1

    for c in fen_parts[2]:
        if c == 'K':
            pos[769] = 1
        elif c == 'Q':
            pos[770] = 1
        elif c == 'k':
            pos[771] = 1
        elif c == 'q':
            pos[772] = 1
    if fen_parts[3] != '-':
        pos[-1] = 1
        
    return pos

In [4]:
X=torch.tensor(X_fen.map(fen_to_pos).to_list())
Y=torch.tensor(Y.to_list())

In [5]:
X = X.float()
Y = Y.float()

In [6]:
n1 = int(0.8 * len(X))
n2 = int(0.9 * len(X))
X_train,Y_train = X[:n1],Y[:n1]
X_dev,Y_dev = X[n1:n2],Y[n1:n2]
X_test,Y_test= X[n2:],Y[n2:]

In [7]:
Y_train

tensor([  27.,   13.,   11.,  ..., -551., -505., -584.])

In [8]:
W1 = torch.rand((774,5000),requires_grad=True)
B1 = torch.rand(5000,requires_grad=True)
W2 = torch.rand((5000,1),requires_grad=True)
B2 = torch.rand(1,requires_grad=True)

In [12]:
X_temp = X_train[:100]
Y_temp = Y_train[:100]